In [1]:
import glob
import os
import numpy as np
import pandas as pd

folder = r"C:\Users\ASUS\Desktop\volPredict"


def load_dataset_type(file_pattern, name):
    files = glob.glob(os.path.join(folder, file_pattern))
    if not files:
        print(f"⚠️ Warning: No files found for pattern: {file_pattern}")
        return pd.DataFrame()

    df_list = []
    for f in files:
        df_temp = pd.read_csv(f, low_memory=False)
        df_temp.columns = df_temp.columns.str.strip()
        df_list.append(df_temp)

    df_combined = pd.concat(df_list, ignore_index=True)
    print(
        f"✓ Loaded {name:<10}: {len(files)} files | {len(df_combined):,} total rows"
    )
    return df_combined


print("=== INGESTING RAW NSE FILES ===")
ce = load_dataset_type("OPTIDX_NIFTY_CE_*.csv", "Calls (CE)")
pe = load_dataset_type("OPTIDX_NIFTY_PE_*.csv", "Puts (PE)")
fut = load_dataset_type("FUTIDX_NIFTY_*.csv", "Futures")
vix = load_dataset_type("hist_india_vix_*.csv", "India VIX")

=== INGESTING RAW NSE FILES ===
✓ Loaded Calls (CE): 18 files | 1,040,344 total rows
✓ Loaded Puts (PE) : 18 files | 1,075,736 total rows
✓ Loaded Futures   : 18 files | 3,404 total rows
✓ Loaded India VIX : 5 files | 1,113 total rows


In [2]:
print("=== CLEANING DATES & NUMERIC FIELDS ===")

# 1. Parse Dates safely
for df, label in [(ce, "CE"), (pe, "PE"), (fut, "FUT")]:
    if not df.empty:
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
        df["Expiry"] = pd.to_datetime(
            df["Expiry"], dayfirst=True, errors="coerce"
        )

if not vix.empty:
    vix["Date"] = pd.to_datetime(vix["Date"], dayfirst=True, errors="coerce")

# 2. Columns to convert to float (handling '-' as NaN)
opt_cols = [
    "Strike Price",
    "Open",
    "High",
    "Low",
    "Close",
    "LTP",
    "Settle Price",
    "No. of contracts",
    "Open Int",
    "Change in OI",
    "Underlying Value",
]

for col in opt_cols:
    if col in ce.columns:
        ce[col] = pd.to_numeric(ce[col], errors="coerce")
    if col in pe.columns:
        pe[col] = pd.to_numeric(pe[col], errors="coerce")

fut_cols = [
    "Open",
    "High",
    "Low",
    "Close",
    "LTP",
    "Settle Price",
    "No. of contracts",
    "Open Int",
    "Change in OI",
    "Underlying Value",
]
for col in fut_cols:
    if col in fut.columns:
        fut[col] = pd.to_numeric(fut[col], errors="coerce")

vix_cols = ["Open", "High", "Low", "Close", "Prev. Close", "Change", "% Change"]
for col in vix_cols:
    if col in vix.columns:
        vix[col] = pd.to_numeric(vix[col], errors="coerce")

print("✓ Numeric conversions completed successfully.")

=== CLEANING DATES & NUMERIC FIELDS ===


C:\Users\ASUS\AppData\Local\Temp\ipykernel_62052\4149641353.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  vix["Date"] = pd.to_datetime(vix["Date"], dayfirst=True, errors="coerce")


✓ Numeric conversions completed successfully.


In [3]:
print("=== APPLYING DATE CUTOFF & VERIFYING DATA ===")

end_date = pd.Timestamp("2025-06-30")

ce = ce[ce["Date"] <= end_date].copy()
pe = pe[pe["Date"] <= end_date].copy()
fut = fut[fut["Date"] <= end_date].copy()
vix = vix[vix["Date"] <= end_date].copy()


def print_summary(df, name):
    if df.empty:
        print(f"{name:<10}: Empty DataFrame")
        return
    min_d = df["Date"].min().strftime("%Y-%m-%d")
    max_d = df["Date"].max().strftime("%Y-%m-%d")
    unique_days = df["Date"].nunique()
    print(
        f"{name:<10}: {min_d} to {max_d} | Trading Days: {unique_days} | Rows: {len(df):,}"
    )


print_summary(ce, "CE Options")
print_summary(pe, "PE Options")
print_summary(fut, "Futures")
print_summary(vix, "India VIX")

=== APPLYING DATE CUTOFF & VERIFYING DATA ===
CE Options: 2021-01-01 to 2025-06-30 | Trading Days: 1113 | Rows: 1,040,344
PE Options: 2021-01-01 to 2025-06-30 | Trading Days: 1113 | Rows: 1,075,736
Futures   : 2021-01-01 to 2025-06-30 | Trading Days: 1113 | Rows: 3,404
India VIX : 2021-01-01 to 2025-06-30 | Trading Days: 1113 | Rows: 1,113
